# Bronze — Elevação do Terreno (Open-Meteo Elevation API)

Ingestão direta da API pública Open-Meteo Elevation, sem depender de
arquivo intermediário na landing zone antes da chamada. A API aceita
lotes de até 100 pares de coordenadas por requisição e retorna a
elevação do terreno (metros, datum WGS84) com resolução de 90 m
(Copernicus DEM GLO-90). Não há parâmetro de data/hora — o dado é
estático (elevação não muda por execução).

**Responsável:** Adenilson Gomes  
**Frequência de execução:** Carga inicial (dado estático)


## Bloco 1 — Importações

A lógica de chamada HTTP (com retry e backoff) vive em
`elevation_client.py`, fora do notebook. O notebook só orquestra:
chama a função, valida, pousa no Volume, grava na Delta Table.

Carrega as coordenadas dos 94 bairros a partir do JSON disponibilizado pela Prefeitura do Recife

In [0]:
import sys, importlib, json, os
sys.path.append("/Workspace/Users/gabriel.fo.br@gmail.com/AlagarIA-IA_Project/APIs")
 
import elevation_client
importlib.reload(elevation_client)
from elevation_client import fetch_elevation_data
from pyspark.sql import functions as F
from datetime import datetime, timezone
 
NOME_FONTE_ELEVATION = "open_meteo_elevation"
CATALOGO             = "alerta_alagamento_recife"
SCHEMA_BRONZE        = "bronze"
data_execucao        = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S")
PRIMEIRA_CARGA       = True
 
# Carrega coordenadas dos 94 bairros a partir do JSON 
CAMINHO_BAIRROS = "/Workspace/Users/gabriel.fo.br@gmail.com/AlagarIA-IA_Project/APIs/bairros_recife_coords.json"
CAMINHO_RPA     = "/Workspace/Users/gabriel.fo.br@gmail.com/AlagarIA-IA_Project/APIs/bairros_rpa.json"
 
with open(CAMINHO_BAIRROS, encoding="utf-8") as f:
    bairros = json.load(f)

with open(CAMINHO_RPA, encoding="utf-8") as f:
    dados_rpa = json.load(f)

bairro_para_rpa = {
    record[1].strip().title(): record[2]
    for record in dados_rpa["records"]
}
 
BAIRROS    = [b["bairro"]    for b in bairros]   #nomes (para auditoria)
LATITUDES  = [b["latitude"]  for b in bairros]
LONGITUDES = [b["longitude"] for b in bairros]
 
print(f"✓ {len(bairros)} bairros carregados do JSON.")
print(f"  Exemplo: {bairros[0]}")
 
#  A API aceita no máximo 100 coords por chamada 
# Se len(bairros) > 100, dividimos em lotes (não é o caso dos 94 bairros,
# mas boa prática para não quebrar se a lista crescer).

assert len(LATITUDES) <= 100, (
    f"A API aceita até 100 coords por chamada; há {len(LATITUDES)} bairros. "
    "Implemente paginação em lotes de 100."
)

## Bloco 2 — Execução da chamada à API (com retry já embutido)

`fetch_elevation_data` já tenta até 3 vezes com backoff antes de
devolver falha. Aqui no notebook só tratamos o resultado final:
se mesmo com as tentativas a API não respondeu, registramos o
problema e seguimos sem gravar nada nesta execução — isso é
melhor do que travar o Job esperando indefinidamente.

O endpoint é:
```
GET https://api.open-meteo.com/v1/elevation
    ?latitude=<lat1,lat2,...>
    &longitude=<lon1,lon2,...>
```
Resposta de sucesso: `{ "elevation": [<float>, ...] }` — sempre um
array, mesmo para coordenada única.

In [0]:
resultado = fetch_elevation_data(latitudes=LATITUDES, longitudes=LONGITUDES)

if not resultado["sucesso"]:
    print(
        f"[ALERTA] Falha na API Open-Meteo Elevation após {resultado['tentativas']} "
        f"tentativa(s): {resultado['erro']}"
    )

## Bloco 3 — Governança: validação de schema antes de gravar

Checamos se o campo `elevation` existe na resposta e se o número de
valores retornados bate com o número de coordenadas enviadas. Se a
Open-Meteo mudar o formato do retorno, o pipeline falha aqui, de
forma visível — não silenciosamente mais na frente, na Silver.

In [0]:
CAMPOS_ESPERADOS_ELEVATION = {"elevation"}


def validar_schema_elevation(
    dados: dict,
    campos_esperados: set,
    n_coords: int,
    nome_fonte: str,
) -> None:
    """
    Confere se a resposta contém o campo 'elevation' e se o número
    de valores retornados é igual ao número de coordenadas enviadas.
    Não corrige nada — só interrompe a execução com mensagem clara
    se o schema da fonte tiver mudado.
    """
    if not dados:
        raise RuntimeError(f"[{nome_fonte}] resposta da API vazia, nada para validar.")

    campos_recebidos = set(dados.keys())
    campos_faltando  = campos_esperados - campos_recebidos

    if campos_faltando:
        raise RuntimeError(
            f"[{nome_fonte}] schema mudou — campos ausentes na resposta da API: "
            f"{campos_faltando}. Pipeline interrompido para evitar gravar dado incompleto."
        )

    n_retornados = len(dados["elevation"])
    if n_retornados != n_coords:
        raise RuntimeError(
            f"[{nome_fonte}] inconsistência de cardinalidade: enviamos {n_coords} "
            f"coordenada(s) mas recebemos {n_retornados} valor(es) de elevação."
        )


if resultado["sucesso"]:
    dados_elevation = resultado["dados"]  # dict: {"elevation": [...]}

    validar_schema_elevation(
        dados_elevation,
        CAMPOS_ESPERADOS_ELEVATION,
        len(LATITUDES),
        NOME_FONTE_ELEVATION,
    )
    print(f"✓ Schema validado — {len(dados_elevation['elevation'])} valor(es) de elevação recebido(s).")
else:
    dados_elevation = None

## Bloco 4 — Normaliza para lista de registros e pousa o JSON bruto no Volume

A API retorna `{ "elevation": [v1, v2, ...] }` — um array achatado.
Antes de pousar, montamos uma lista de dicts `{latitude, longitude,
elevation_m}` para que o Spark consiga ler como linhas independentes
e para preservar o vínculo coordenada ↔ valor.

Salvamos o JSON normalizado no Volume antes de processar com Spark:
se algo der errado mais adiante, dá para reprocessar a partir desse
arquivo sem chamar a API de novo.

In [0]:
if dados_elevation is not None:
    # Normaliza para lista de registros (vincula coordenada ↔ elevação)
    registros_elevation = [
        {
            "bairro"         : bairro,
            "rpa"            : bairro_para_rpa.get(bairro.strip().title()),
            "latitude"       : lat,
            "longitude"      : lon,
            "elevacao_metros": elev,
        }
        for bairro, lat, lon, elev in zip(BAIRROS, LATITUDES, LONGITUDES, dados_elevation["elevation"])
    ]

    caminho_landing_elevation = (
        f"/Volumes/{CATALOGO}/{SCHEMA_BRONZE}/landing_zone/"
        f"open_meteo_elevation/{data_execucao}.json"
    )

    # Garante que o diretório existe
    os.makedirs(os.path.dirname(caminho_landing_elevation), exist_ok=True)

    # Escrita direta no Volume (sem dbutils.fs)
    with open(caminho_landing_elevation, "w", encoding="utf-8") as f:
        json.dump(registros_elevation, f, ensure_ascii=False)

    print(f"✓ JSON normalizado pousado em: {caminho_landing_elevation}")

## Bloco 5 — Lê como DataFrame Spark

Como a chamada já é feita com as coordenadas específicas de interesse
(Bloco 1), não há necessidade de filtro adicional por município —
todos os registros são dos pontos de Recife que queremos. Mantemos
apenas uma verificação de contagem para confirmar integridade.

In [0]:
if PRIMEIRA_CARGA:   #condição para a primeira carga, se for a primeira carga, ele vai ler o json do volume
    if caminho_landing_elevation is not None:
        df_bronze_elevation = spark.read.json(caminho_landing_elevation)

        n = df_bronze_elevation.count()
        print(f"✓ {n} registro(s) lidos do Volume.")

        if n != len(LATITUDES):
            raise RuntimeError(
                f"[ALERTA] Contagem inesperada: esperávamos {len(LATITUDES)} "
                f"registro(s), Spark leu {n}."
            )
    else:
        df_bronze_elevation = None

## Bloco 6 — Metadados de auditoria e gravação na Bronze

Particionado por `_data_execucao`, com `mergeSchema=true` para
tolerar campos novos que a API venha a adicionar no futuro sem
quebrar a escrita.

In [0]:
if PRIMEIRA_CARGA:   
    if df_bronze_elevation is not None:
        df_bronze_elevation = (
            df_bronze_elevation
            .withColumn("_ingerido_em",   F.current_timestamp())
            .withColumn("_fonte",         F.lit(NOME_FONTE_ELEVATION))
            .withColumn("_data_execucao", F.lit(data_execucao))
        )

        (
            df_bronze_elevation.write
            .format("delta")
            .mode("append")
            .partitionBy("_data_execucao")
            .option("mergeSchema", "true")
            .saveAsTable(f"{CATALOGO}.{SCHEMA_BRONZE}.open_meteo_elevation")
        )

        spark.sql(f"""
            COMMENT ON TABLE {CATALOGO}.{SCHEMA_BRONZE}.open_meteo_elevation IS
            'Elevação do terreno (metros, WGS84) por par de coordenadas, dado cru
            sem transformação. Fonte: api.open-meteo.com/v1/elevation
            (Copernicus DEM GLO-90, resolução 90 m). Ingestão via API com retry/backoff.
            Particionado por _data_execucao.'
        """)

        print(
            f"OK — {df_bronze_elevation.count()} registros gravados em "
            f"{CATALOGO}.{SCHEMA_BRONZE}.open_meteo_elevation"
        )
        df_bronze_elevation.display()
    else:
        print("Nenhum registro gravado nesta execução — API indisponível após retries.")

## Bloco 7 — Log de auditoria da execução

Mesmo padrão dos outros notebooks: toda execução é registrada,
com sucesso ou falha, para observabilidade do pipeline.

In [0]:
log_execucao = spark.createDataFrame([{
    "notebook":          "01_bronze_open_meteo_elevation",
    "timestamp_execucao": datetime.now(timezone.utc).isoformat(),
    "sucesso":           resultado["sucesso"],
    "tentativas":        resultado["tentativas"],
    "erro":              resultado["erro"] if resultado["erro"] is not None else "",
    "linhas_gravadas":   df_bronze_elevation.count() if df_bronze_elevation is not None else 0,
}])

(
    log_execucao.write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{CATALOGO}.governanca.log_ingestao")
)